<a href="https://colab.research.google.com/github/munnurumahesh03-coder/kaggle-predicting-loan-payback/blob/main/07_Ensembling_and_Stacking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Load All OOF and Test Prediction Files

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# --- Define File Paths ---
# It's good practice to define paths at the top.
# You will need to add these notebooks as data sources to the new notebook.

# From Notebook 02 (for the true target)
TRAIN_TARGET_PATH = '/kaggle/input/02-feature-engineering-ipynb/train_featured_v2.csv'

# From Notebook 03
LGBM_OOF_PATH = '/kaggle/input/03-base-models-tuning-ipynb/boosting_oof_preds.csv'
LGBM_TEST_PATH = '/kaggle/input/03-base-models-tuning-ipynb/boosting_test_preds.csv'
LOGREG_TEST_PATH = '/kaggle/input/03-base-models-tuning-ipynb/submission_LogisticRegression.csv'

# From Notebook 04
RF_OOF_PATH = '/kaggle/input/04-random-forest-lgbm-rf-tuning-ipynb/oof_preds_rf.csv'
RF_TEST_PATH = '/kaggle/input/04-random-forest-lgbm-rf-tuning-ipynb/test_preds_rf.csv'

# From Notebook 05
XGB_OOF_PATH = '/kaggle/input/05-xgboost-tuning-ipynb/oof_preds_xgb.csv'
XGB_TEST_PATH = '/kaggle/input/05-xgboost-tuning-ipynb/test_preds_xgb.csv'

# From Notebook 06
CATBOOST_TEST_PATH = '/kaggle/input/06-catboost-tuning-ipynb/submission_catboost.csv'


# --- Load OOF Predictions for Stacking ---
print("--- Loading OOF predictions for Stacking ---")
oof_lgbm = pd.read_csv(LGBM_OOF_PATH)
oof_rf = pd.read_csv(RF_OOF_PATH)
oof_xgb = pd.read_csv(XGB_OOF_PATH)
print("✅ OOF files loaded.")

# --- Load Test Predictions for Blending and Stacking ---
print("\n--- Loading Test predictions for Ensembling ---")
test_lgbm = pd.read_csv(LGBM_TEST_PATH)
test_logreg = pd.read_csv(LOGREG_TEST_PATH)
test_rf = pd.read_csv(RF_TEST_PATH)
test_xgb = pd.read_csv(XGB_TEST_PATH)
test_catboost = pd.read_csv(CATBOOST_TEST_PATH)
print("✅ Test prediction files loaded.")

# --- Load True Target Variable ---
print("\n--- Loading true target variable ---")
train_labels = pd.read_csv(TRAIN_TARGET_PATH, usecols=['loan_paid_back'])
print("✅ True target labels loaded.")


In [ ]:
# Cell 2: Assemble and Analyze Meta-Training Data (Final, Robust Version)

print("--- Assembling Meta-Training DataFrame ---")

# 1. Select ONLY the prediction columns and rename them.
# This is robust to extra columns like 'id' or 'Unnamed: 0' in the CSVs.
# We assume the prediction is the last column in each file.
lgbm_preds = oof_lgbm.iloc[:, -1].rename('lgbm_pred')
rf_preds = oof_rf.iloc[:, -1].rename('rf_pred')
xgb_preds = oof_xgb.iloc[:, -1].rename('xgb_pred')

# 2. Create the meta-training DataFrame by concatenating the prediction Series.
# We use the 'train_labels' DataFrame loaded in Cell 1.
meta_df = pd.concat([lgbm_preds, rf_preds, xgb_preds, train_labels], axis=1)

print("✅ Meta-DataFrame created successfully:")
print(meta_df.head())


# 3. Evaluate the performance of each base model using their OOF predictions.
print("\n--- Individual OOF Model Scores (AUC) ---")
# Import metrics and visualization libraries here, as they are only used from this cell onwards
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')

lgbm_oof_score = roc_auc_score(meta_df['loan_paid_back'], meta_df['lgbm_pred'])
rf_oof_score = roc_auc_score(meta_df['loan_paid_back'], meta_df['rf_pred'])
xgb_oof_score = roc_auc_score(meta_df['loan_paid_back'], meta_df['xgb_pred'])

print(f"   LightGBM OOF Score: {lgbm_oof_score:.6f}")
print(f"   Random Forest OOF Score: {rf_oof_score:.6f}")
print(f"   XGBoost OOF Score: {xgb_oof_score:.6f}")


# 4. Analyze the correlation between the base models' predictions.
print("\n--- OOF Prediction Correlation Matrix ---")
correlation_matrix = meta_df[['lgbm_pred', 'rf_pred', 'xgb_pred']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.4f')
plt.title('Correlation Matrix of OOF Predictions')
plt.show()

print("\n✅ Analysis complete. Ready for blending and stacking.")


In [ ]:
# Cell 3: Blending - Simple and Weighted Averaging (Robust Version)

print("--- Method 1: Blending ---")

# 1. Assemble the test predictions into a single DataFrame for easier handling.
# This is robust to extra columns like 'id' or 'Unnamed: 0' in the CSVs.

# We need the 'id' column for the final submission. We'll take it from one file.
submission_ids = test_logreg.iloc[:, 0]

# Select ONLY the prediction column from each file (assuming it's the last one).
logreg_preds = test_logreg.iloc[:, -1].rename('logreg_pred')
lgbm_preds = test_lgbm.iloc[:, -1].rename('lgbm_pred')
rf_preds = test_rf.iloc[:, -1].rename('rf_pred')
xgb_preds = test_xgb.iloc[:, -1].rename('xgb_pred')
catboost_preds = test_catboost.iloc[:, -1].rename('catboost_pred')


# 2. Simple Average Blend
print("\n--- Creating Simple Average Blend ---")
# We will blend all 5 models.
blend_simple_avg = (logreg_preds +
                    lgbm_preds +
                    rf_preds +
                    xgb_preds +
                    catboost_preds) / 5

# Create the submission file for the simple blend
submission_blend_simple = pd.DataFrame({'id': submission_ids, 'loan_paid_back': blend_simple_avg})
submission_blend_simple.to_csv('submission_blend_simple_avg.csv', index=False)
print("✅ 'submission_blend_simple_avg.csv' created.")
print(submission_blend_simple.head())


# 3. Weighted Average Blend
print("\n--- Creating Weighted Average Blend ---")
# Weights based on OOF scores (higher score = higher weight)
# XGB and LGBM are best, CatBoost is good, RF is next, LogReg is baseline.
weights = {
    'xgb': 0.35,
    'lgbm': 0.35,
    'cat': 0.15, # We don't have an OOF score, but we know it's a strong model.
    'rf': 0.10,
    'logreg': 0.05
}

blend_weighted_avg = (weights['xgb'] * xgb_preds +
                      weights['lgbm'] * lgbm_preds +
                      weights['cat'] * catboost_preds +
                      weights['rf'] * rf_preds +
                      weights['logreg'] * logreg_preds)

# Create the submission file for the weighted blend
submission_blend_weighted = pd.DataFrame({'id': submission_ids, 'loan_paid_back': blend_weighted_avg})
submission_blend_weighted.to_csv('submission_blend_weighted_avg.csv', index=False)
print("✅ 'submission_blend_weighted_avg.csv' created.")
print(submission_blend_weighted.head())


In [ ]:
# Cell 4: Stacking - Training the Meta-Model

print("--- Method 2: Stacking ---")

# 1. Define our Meta-Features (X_meta) and Meta-Target (y_meta)
# The features are the OOF predictions from our base models.
X_meta = meta_df[['lgbm_pred', 'rf_pred', 'xgb_pred']]
y_meta = meta_df['loan_paid_back']

print("Meta-training data (X_meta) head:")
print(X_meta.head())


# 2. Define the Meta-Model.
# A simple, stable Logistic Regression is an excellent choice for a meta-model.
# It's less likely to overfit the OOF predictions.
from sklearn.linear_model import LogisticRegression
meta_model = LogisticRegression()


# 3. Train the Meta-Model on the entire OOF dataset.
print("\n--- Training Meta-Model (Logistic Regression) ---")
meta_model.fit(X_meta, y_meta)
print("✅ Meta-Model trained successfully.")


# 4. Assemble the test data for the meta-model.
# The features must be in the EXACT same order as the training data.
# We'll reuse the prediction Series we created in Cell 3.
X_meta_test = pd.concat([lgbm_preds, rf_preds, xgb_preds], axis=1)

# Ensure column names match the training data for predict()
X_meta_test.columns = ['lgbm_pred', 'rf_pred', 'xgb_pred']

print("\nMeta-test data (X_meta_test) head:")
print(X_meta_test.head())


# 5. Use the trained meta-model to make final predictions on the test set.
print("\n--- Generating Final Stacked Predictions ---")
stacked_predictions = meta_model.predict_proba(X_meta_test)[:, 1]


# 6. Create and save the final stacking submission file.
submission_stacking = pd.DataFrame({'id': submission_ids, 'loan_paid_back': stacked_predictions})
submission_stacking.to_csv('submission_stacking.csv', index=False)
print("✅ 'submission_stacking.csv' created.")
print(submission_stacking.head())
